# dynamic_vacuum_sim — Quickstart Notebook

This notebook walks through every major feature of `dynamic_vacuum_sim`, a Python package that computes hydrogenic spectra using:

1. **Standard Rydberg model** — the familiar Coulomb/Schrödinger 1/n² ladder.
2. **Dynamic-vacuum acoustic framework** — the mapping from [White et al., *Phys. Rev. Research* **8**, 013264 (2026)](https://doi.org/10.1103/l8y7-r3rm).

Both models produce identical results (exact isospectrality), demonstrated below.

---

## Setup

```bash
# Run once from the repo root:
pip install -e ".[dev]"
```

In [ ]:
from dynamic_vacuum_sim import rydberg, dynamic_vacuum as dv, radial, plotting, constants as C
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Energy Levels

Compute hydrogen bound-state energies using the reduced-mass Rydberg constant $R_H$ (CODATA-2018):

$$|E_n| = h\,c\,R_H \;/\; n^2$$

In [ ]:
# Single level
rydberg.level_energy(1)

In [ ]:
# All levels n=1..7
for n in range(1, 8):
    lv = rydberg.level_energy(n)
    print(f"n={n}:  {lv['energy_eV']:12.6f} eV   {lv['frequency_PHz']:.6f} PHz")

## 2. Spectral Lines & Series

Transition frequencies and vacuum wavelengths, using Eq. (20) of the paper:

$$f_{n_2 \to n_1} = c\,R_H\left(\frac{1}{n_1^2} - \frac{1}{n_2^2}\right)$$

In [ ]:
# Lyman-alpha
rydberg.transition(n_upper=2, n_lower=1)

In [ ]:
# Full Balmer series
for line in rydberg.series("balmer", n_max=10):
    print(f"  {line['n_upper']} → {line['n_lower']}:  λ = {line['wavelength_nm']:.3f} nm")

## 3. Dynamic-Vacuum Mapping

The acoustic framework adds physical interpretation to the same mathematics.  Key quantities:

| Symbol | Equation | Meaning |
|--------|----------|---------|
| $\kappa_n = \beta/(2n)$ | Eq. 18 | Bound-state inverse decay length |
| $\omega_n = D\kappa_n^2 = \omega_*/n^2$ | Eq. 10 | Eigenfrequency |
| $E_n = -\hbar\omega_n$ | Eq. 19 | Energy (identical to Rydberg) |
| $A(\omega_n) < 0$ | Eq. 22 | Reactive stop band (→ localization) |
| $C(\omega_n) > 0$ | Eq. 22 | Proton-induced coupling |

In [ ]:
# Dynamic-vacuum level data includes kappa and omega
dv.level_energy(1)

In [ ]:
# Constitutive coefficients
for n in range(1, 6):
    print(f"n={n}:  A(ω_n) = {dv.A_coeff(n):.6e}   C(ω_n) = {dv.C_coeff(n):.6e}")

## 4. Isospectrality Verification

The central claim of the paper: both models give **identical** spectra.  The function below compares energies and raises an error if they diverge beyond a tolerance.

In [ ]:
results = dv.verify_isospectrality(n_max=10, rtol=1e-12)

for r in results:
    print(f"  n={int(r['n']):2d}:  E_rydberg = {r['E_rydberg_eV']:.9f} eV,  "
          f"E_dv = {r['E_dv_eV']:.9f} eV,  rel_err = {r['rel_error']:.1e}")

print("\n✓ Exact match to machine precision.")

## 5. Radial Wave Functions

Hydrogenic radial eigenfunctions $R_{n\ell}(r)$ from Eq. (13a):

$$R_{n\ell}(r) = \mathcal{N}_{n\ell}\;(2\kappa_n r)^\ell \; e^{-\kappa_n r}\; L^{2\ell+1}_{n-\ell-1}(2\kappa_n r)$$

In [ ]:
r = np.linspace(1e-15, 25 * C.A0, 1000)
r_bohr = r / C.A0

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (n, ell) in zip(axes, [(1, 0), (2, 1), (3, 2)]):
    density = radial.radial_probability_density(n, ell, r) * C.A0
    ax.plot(r_bohr, density)
    ax.fill_between(r_bohr, density, alpha=0.15)
    ax.set_title(f"n={n}, ℓ={ell}")
    ax.set_xlabel(r"$r\;/\;a_0$")
    ax.set_xlim(0, 25)
    ax.set_ylim(bottom=0)

axes[0].set_ylabel(r"$r^2 |R_{n\ell}|^2$  (per $a_0$)")
fig.suptitle("Radial Probability Densities", fontsize=13)
fig.tight_layout()
plt.show()

## 6. Built-in Plotting Helpers

The `plotting` module provides ready-made figures.

In [ ]:
# Energy level ladder
fig = plotting.plot_levels(n_max=7)
plt.show()

In [ ]:
# Spectral series (Grotrian diagram)
fig = plotting.plot_series()
plt.show()

In [ ]:
# Rydberg vs. dynamic-vacuum overlay — visual proof of isospectrality
fig = plotting.plot_radial_comparison(n=3, ell=1)
plt.show()

## 7. Full Madelung Dispersion (Appendix)

The complete dispersion relation from the paper's Appendix, Eq. (A21):

$$\omega^2 = c_L^2\,k^2 + D^2\,k^4$$

In the quantum-pressure limit ($c_L \to 0$), this reduces to $\omega = Dk^2$.

In [ ]:
k = np.linspace(0.01e10, 5e10, 500)

fig, ax = plt.subplots(figsize=(7, 4))
for c_L, label in [(0.0, r"$c_L = 0$ (quadratic)"), (1e4, r"$c_L = 10^4$ m/s")]:
    omega = np.array([dv.dispersion_full(ki, c_L=c_L) for ki in k])
    ax.plot(k / 1e10, omega / 1e15, label=label)

ax.set_xlabel(r"$k$  ($10^{10}$ m$^{-1}$)")
ax.set_ylabel(r"$\omega$  (PHz)")
ax.set_title("Madelung Dispersion (Eq. A21)")
ax.legend()
fig.tight_layout()
plt.show()

## 8. Orthonormality Check

Verify that $\langle R_{n'\ell} | R_{n\ell} \rangle = \delta_{n'n}$ numerically.

In [ ]:
results = radial.verify_orthonormality(n_max=4, r_max_bohr=200, n_pts=10_000)

for entry in results:
    expected = 1.0 if entry['n1'] == entry['n2'] else 0.0
    marker = "✓" if entry['passed'] else "✗"
    print(f"  ⟨R_{{{entry['n1']},{entry['ell']}}}|R_{{{entry['n2']},{entry['ell']}}}⟩"
          f" = {entry['overlap']:+.6f}  (expected {expected:.0f})  {marker}")

## 9. Physical Constants

All CODATA-2018 values and derived quantities are available in `constants`.

In [ ]:
print(f"Reduced mass μ         = {C.MU:.10e} kg")
print(f"Bohr radius a₀(H)     = {C.A0:.10e} m")
print(f"Rydberg R_H            = {C.R_H:.6f} m⁻¹")
print(f"ω*                     = {C.OMEGA_STAR:.6e} rad/s")
print(f"Dispersion D           = {C.D_DISP:.6e} m²/s")
print(f"β = 2/a₀              = {C.BETA:.6e} m⁻¹")
print(f"Rydberg energy         = {C.RY_EV:.9f} eV")

---

## CLI Usage (from terminal)

The package also ships a `dv-hydrogen` command:

```bash
dv-hydrogen level --n 1 --n 2 --n 3 --n 4 --n 5 --n 6 --n 7
dv-hydrogen line --n1 1 --n2 3
dv-hydrogen verify --nmax 7
dv-hydrogen plot radial --n 3 --ell 2 -o radial.png
dv-hydrogen plot levels --nmax 7 -o levels.png
dv-hydrogen plot series -o series.png
```

---

**Reference:**  
White, Vera, Sylvester & Dudzinski, "Emergent quantization from a dynamic vacuum",  
*Phys. Rev. Research* **8**, 013264 (2026).  
[DOI: 10.1103/l8y7-r3rm](https://doi.org/10.1103/l8y7-r3rm)